#### Метод `rollback()`

`rollback()` — это метод для отката транзакции: он отменяет все изменения в базе данных, сделанные в рамках текущей транзакции, и возвращает базу в состояние, которое было до её начала.

`Ключевые моменты`:

- `Транзакция` — это атомарный блок операций. Принцип: «всё или ничего». Либо выполняются все команды в транзакции, либо ни одной.

- Если одна из SQL‑команд внутри транзакции падает с ошибкой, СУБД помечает транзакцию как «неудачную». Дальнейшие команды в этой транзакции либо не выполняются, либо сразу возвращают ошибку, потому что транзакция уже в состоянии сбоя.

- Чтобы «сбросить» это состояние и вернуть соединение к нормальной работе, нужно явно вызвать `rollback()`. После этого соединение готово к новой транзакции.

#### Пример на `Python` + `PostgreSQL (psycopg3)`

```python
import psycopg

conn = psycopg.connect("dbname=hexlet_test user=postgres password=secret")

# ОБРАТИ ВНИМАНИЕ!!
conn.autocommit = False  # отключаем автокоммит, чтобы управлять транзакциями вручную

try:
    with conn.cursor() as cur:
        cur.execute("UPDATE users SET balance = balance - 100 WHERE id = 1")
        # Допустим, здесь происходит ошибка (например, неверный тип данных, нарушение ограничения и т.п.)
        cur.execute("UPDATE accounts SET total = total + 100 WHERE user_id = 1")  # может вызвать исключение
    conn.commit()  # если всё прошло успешно — фиксируем изменения
except Exception:
    conn.rollback()  # откатываем все изменения, сделанные в этой транзакции
    print("Транзакция отменена")
```

Если не вызывать `rollback()`, соединение осталось бы в состоянии «транзакция в ошибке», и следующие SQL‑запросы могли бы не выполниться или сразу возвращать ошибку транзакции.

#### `Важные нюансы`

1. Разница между `commit()` и `rollback()`:

    - `commit()` — сохраняет все изменения транзакции навсегда.

    - `rollback()` — полностью отменяет все изменения транзакции.

2. `Автокоммит`. Если включён автокоммит `(autocommit=True)`, каждая команда выполняется как отдельная транзакция и сразу фиксируется. В таком режиме `rollback()` не имеет смысла, потому что изменения уже зафиксированы.

3. `Контекстные менеджеры`. В современных драйверах (например, `psycopg`) удобно использовать `with` для управления транзакциями: при выходе из блока без ошибок автоматически делается `commit()`, а при исключении — `rollback()`.

```python

with psycopg.connect("dbname=hexlet_test") as conn:
    with conn.cursor() as cur:
        cur.execute("UPDATE users SET balance = balance - 100 WHERE id = 1")
        cur.execute("UPDATE accounts SET total = total + 100 WHERE user_id = 1")
# Если здесь не было исключений — будет commit, иначе — rollback
```